# Music Annotation Project — Notebook 38.5
## V8.2i Envelope Relaxation and Perceptual Severity Calibration

**Runtime:** Colab CPU. Run all cells in order.

**Status:** development-only, single-reviewer exploratory. No retraining, recalibration, threshold adoption, challenger freeze, independent-test nomination or production authority.

### Amendment 38.5
This complete replacement for Notebook 38.4 deduplicates component rows to one record per page and adjacent-threshold transition, then collapses byte-identical whole-page pairs. It uses Notebook 37's persisted pages, provides a fully labelled guided reviewer, locks the blind first pass before diagnostic reveal, resumes from the next incomplete pair and maps each unique review back to every represented transition.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pillow matplotlib


## 1. Configuration and helpers


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import base64,hashlib,json,os,shutil,zipfile
import numpy as np,pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier';OUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None;ASSET_ROOT_OVERRIDE=None;SEED=38001;REVIEWER_ID='reviewer_01';VERSION='38.5-guided-deduplicated-review'
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_38_5';OUT=OUT_ROOT/RUN;TABLES=OUT/'tables';REVIEW=OUT/'review';AUDIT=OUT/'audit';VIS=OUT/'visuals';REF=OUT/'reference';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,VIS,REF]:p.mkdir(parents=True,exist_ok=False)
def now():return datetime.now(timezone.utc).isoformat()
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def text(p,s):p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');q.write_text(s,encoding='utf-8');os.replace(q,p)
def js(p,x):text(p,json.dumps(x,indent=2,sort_keys=True,default=str))
def csv(d,p):p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');d.to_csv(q,index=False);os.replace(q,p)
def stop(stage,msg,details=None):js(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':msg,'details':details or {},'utc':now()});raise RuntimeError(f'{stage}: {msg}')
def col(d,names,required=True):
 z={str(x).lower():x for x in d.columns}
 for n in names:
  if n.lower() in z:return z[n.lower()]
 if required:raise KeyError(f'Missing one of {names}; columns={list(d.columns)}')
def image_ok(p):
 try:
  with Image.open(p) as im:im.verify()
  return True
 except:return False
def latest(root,required):
 hits=[p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in required)] if Path(root).exists() else []
 return sorted(hits,key=lambda p:p.stat().st_mtime)[-1] if hits else None
print('Output:',OUT)


## 2. Resolve Notebook 37, lock inputs and audit its package manifest


In [ ]:
REQ=['experiment_identity.json','source_v82_record.json','tables/threshold_frontier_summary.csv','tables/incremental_component_review_queue_all.csv','tables/catastrophic_component_first_appearance_inventory.csv','upload/upload_manifest.csv']
NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else latest(NB37_ROOT,REQ)
if NB37 is None:stop('input','No complete Notebook 37 run found',{'root':str(NB37_ROOT)})
locks=[]
for r in REQ:
 p=NB37/r
 if not p.is_file():stop('input','Required Notebook 37 file missing',{'file':r})
 locks.append({'relative_path':r,'absolute_path':str(p.resolve()),'bytes':p.stat().st_size,'sha256':sha(p)})
source_lock=pd.DataFrame(locks);csv(source_lock,AUDIT/'notebook38_source_input_lock.csv');source_lock_sha256=hashlib.sha256(source_lock.to_csv(index=False).encode()).hexdigest()
shutil.copy2(NB37/'experiment_identity.json',REF/'notebook37_experiment_identity.json');shutil.copy2(NB37/'source_v82_record.json',REF/'source_v82_record.json')
manifest=pd.read_csv(NB37/'upload/upload_manifest.csv');pc=col(manifest,['relative_path','path','file','filename']);hc=col(manifest,['sha256','sha_256','hash'],False);mr=[]
for _,r in manifest.iterrows():
 raw=str(r[pc]);q=Path(raw);opts=[NB37/'upload'/q,NB37/q]
 if 'upload' in q.parts:
  k=list(q.parts).index('upload');opts.insert(0,NB37/'upload'/Path(*list(q.parts)[k+1:]))
 p=next((x for x in opts if x.is_file()),None);expected=str(r[hc]).strip() if hc and pd.notna(r[hc]) else '';actual=sha(p) if p else '';match=(actual.lower()==expected.lower()) if p and expected else np.nan
 status='verified_package_copy' if p and (not expected or match is True) else ('package_copy_hash_mismatch' if p else 'unresolved_package_row')
 mr.append({'manifest_path':raw,'resolved':str(p or ''),'expected_sha256':expected,'actual_sha256':actual,'status':status})
ma=pd.DataFrame(mr);csv(ma,AUDIT/'nb37_manifest_verification.csv');manifest_summary={'scope':'Notebook 37 curated package only','rows':len(ma),'warnings':int(ma.status.ne('verified_package_copy').sum())};js(AUDIT/'nb37_manifest_verification_summary.json',manifest_summary)
protocol={'version':VERSION,'source_nb37_run':str(NB37),'source_lock_sha256':source_lock_sha256,'reviewer_count':1,'status':'single-reviewer-exploratory','production_authority':False,'threshold_adoption_authority':False};js(OUT/'protocol_lock.json',protocol)
print('Notebook 37:',NB37);print(manifest_summary)


## 3. Registered threshold identity and D3–D8 record


In [ ]:
front=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv');actual=[round(float(x),3) for x in front[col(front,['threshold'])]]
if actual!=THRESHOLDS:stop('thresholds','Threshold identity/order mismatch',{'expected':THRESHOLDS,'actual':actual})
fields=['threshold','true_print','false_annotation','utility','catastrophic_components','false_true_ratio','false_preservation_rate','benefiting_pages','benefiting_source_groups','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability']
missing=[x for x in fields if x not in front]
if missing:stop('registered_record','Missing registered fields',{'missing':missing})
csv(front[fields],TABLES/'registered_D3_D8_results.csv');print(front[fields].to_string(index=False))


## 4. Deduplicate component rows to page × adjacent-threshold transitions


In [ ]:
q=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv')
def opt(names,default=''):
 x=col(q,names,False);return q[x] if x is not None else pd.Series([default]*len(q),index=q.index)
raw=pd.DataFrame({'page':q[col(q,['page'])].astype(str),'source_group':q[col(q,['source_group'])].astype(str),'component':pd.to_numeric(q[col(q,['mask_component_id','component_id'])]).astype(int),'hi':pd.to_numeric(q[col(q,['higher_threshold','from_threshold','high_threshold'])]).round(3),'lo':pd.to_numeric(q[col(q,['lower_threshold','to_threshold','low_threshold'])]).round(3),'new_tp':pd.to_numeric(opt(['new_true_print','incremental_true_print','true_print'],0),errors='coerce').fillna(0),'new_fp':pd.to_numeric(opt(['new_false_annotation','incremental_false_annotation','false_annotation'],0),errors='coerce').fillna(0),'anchor_local':opt(['local_panel','local_crop','local_path']),'anchor_wide':opt(['wide_panel','wide_crop','wide_path']),'anchor_locator':opt(['locator','whole_page_locator','locator_path']),'anchor_truth':opt(['truth_overlay','cumulative_truth_overlay','truth_overlay_path'])})
def first_nonblank(s):
 z=[str(x) for x in s if pd.notna(x) and str(x).strip()];return z[0] if z else ''
transitions=(raw.groupby(['page','source_group','hi','lo'],as_index=False).agg(component_count=('component','nunique'),component_ids=('component',lambda s:';'.join(map(str,sorted(set(s))))),new_tp=('new_tp','sum'),new_fp=('new_fp','sum'),anchor_local=('anchor_local',first_nonblank),anchor_wide=('anchor_wide',first_nonblank),anchor_locator=('anchor_locator',first_nonblank),anchor_truth=('anchor_truth',first_nonblank)))
transitions['transition_id']=transitions.apply(lambda r:'TR_'+hashlib.sha256(f'{r.page}|{r.source_group}|{r.hi:.3f}|{r.lo:.3f}'.encode()).hexdigest()[:16],axis=1)
if transitions.transition_id.duplicated().any():stop('deduplication','Duplicate transition IDs remain')
csv(raw,TABLES/'component_queue_source_rows.csv');csv(transitions,TABLES/'page_transition_manifest.csv')
print('Component rows:',len(raw));print('Unique page-transition rows:',len(transitions));print('Removed:',len(raw)-len(transitions))


## 5. Resolve persisted pages and collapse byte-identical pairs

Authoritative filenames: `threshold_candidate.png`, `v8_2_registered.png`, `v7_8.png`, `observed.png`, `difference_from_registered_v8_2.png`, and `newly_preserved_vs_0995_overlay.png`. The 0.995 endpoint uses `v8_2_registered.png`; other endpoints use `threshold_candidate.png`.


In [ ]:
NAMES={'candidate':'threshold_candidate.png','registered':'v8_2_registered.png','v78':'v7_8.png','observed':'observed.png','difference':'difference_from_registered_v8_2.png','overlay':'newly_preserved_vs_0995_overlay.png'}
ASSET_ROOT=Path(ASSET_ROOT_OVERRIDE) if ASSET_ROOT_OVERRIDE else NB37
all_dirs=sorted({p.parent.resolve() for p in ASSET_ROOT.rglob(NAMES['candidate'])})
# Prefer working artefacts over duplicate copies beneath upload/. Fall back to package copies only when needed.
working_dirs=[d for d in all_dirs if not (NB37/'upload').resolve() in d.parents]
dirs=working_dirs if working_dirs else all_dirs
if not dirs:stop('assets','No threshold_candidate.png files found',{'root':str(ASSET_ROOT)})
def palias(page):
 s=str(page);d=''.join(x for x in s if x.isdigit());a={s.lower(),Path(s).stem.lower()}
 if d:
  n=int(d);a|={str(n),f'{n:02d}',f'{n:03d}',f'page{n}',f'page{n:02d}',f'page{n:03d}',f'page_{n:02d}',f'page_{n:03d}'}
 return a
def talias(t):
 s=f'{float(t):.3f}';return {s,s.replace('.','_'),s.replace('.',''),f't{s}',f't_{s}',f'threshold_{s}',f'threshold_{s.replace(".","_")}'}
def resolve(page,t):
 ranked=[]
 for d in dirs:
  parts=[x.lower() for x in d.parts];joined='/'.join(parts);ps=max([4 if a in parts else 2 if a in joined else 0 for a in palias(page)] or [0]);ts=max([5 if a in parts else 3 if a in joined else 0 for a in talias(t)] or [0])
  if ps and ts:ranked.append((ps+ts,d))
 if not ranked:return None,'no page/threshold directory match',[]
 ranked=sorted(ranked,key=lambda x:(x[0],-len(x[1].parts)),reverse=True);top=ranked[0][0];tops=sorted({d for s,d in ranked if s==top},key=str)
 return (tops[0],'',ranked[:5]) if len(tops)==1 else (None,f'ambiguous score {top}: {len(tops)} directories',ranked[:5])
def get(d,name):
 p=Path(d)/name if d else None;return p if p and p.is_file() and image_ok(p) else None
resolved=[];audit=[];fails=[]
for _,r in transitions.iterrows():
 ld,le,lr=resolve(r.page,r.lo);hd,he,hr=(ld,'',[]) if np.isclose(r.hi,.995) else resolve(r.page,r.hi);low=get(ld,NAMES['candidate']);high=get(ld,NAMES['registered']) if np.isclose(r.hi,.995) else get(hd,NAMES['candidate']);reasons=[x for x in [le,he] if x]
 if not low:reasons.append('lower candidate missing/unreadable')
 if not high:reasons.append('higher page missing/unreadable')
 if low and high:
  with Image.open(low) as a,Image.open(high) as b:
   if a.size!=b.size:reasons.append(f'shape mismatch {a.size} vs {b.size}')
 ar={'transition_id':r.transition_id,'page':r.page,'hi':r.hi,'lo':r.lo,'low_dir':str(ld or ''),'high_dir':str(hd or ''),'low_path':str(low or ''),'high_path':str(high or ''),'status':'failed' if reasons else 'verified','reason':' | '.join(reasons),'low_candidates':json.dumps([str(x[1]) for x in lr]),'high_candidates':json.dumps([str(x[1]) for x in hr])};audit.append(ar)
 if reasons:fails.append(ar);continue
 diag={k:get(ld,v) for k,v in NAMES.items() if k!='candidate'};pair_key=hashlib.sha256((str(r.page)+'|'+'|'.join(sorted([sha(low),sha(high)]))).encode()).hexdigest();resolved.append({**r.to_dict(),'low_path':str(low),'high_path':str(high),'pair_key':pair_key,'overlay_path':str(diag.get('overlay') or ''),'difference_path':str(diag.get('difference') or ''),'v78_path':str(diag.get('v78') or ''),'observed_path':str(diag.get('observed') or '')})
csv(pd.DataFrame(audit),AUDIT/'whole_page_asset_resolution.csv');csv(pd.DataFrame(fails),AUDIT/'missing_review_assets.csv')
if fails:stop('assets','Persisted page resolution failed',{'count':len(fails),'examples':fails[:10]})
resolved=pd.DataFrame(resolved);pairs=[];mapping=[]
for (page,key),g in resolved.groupby(['page','pair_key'],sort=True):
 rep=g.sort_values(['hi','lo'],ascending=False).iloc[0];rid='RV_'+key[:16];low_left=int(hashlib.sha256(f'{SEED}|{rid}'.encode()).hexdigest(),16)%2==0;A,B=(rep.low_path,rep.high_path) if low_left else (rep.high_path,rep.low_path);side={'A':'lower','B':'higher'} if low_left else {'A':'higher','B':'lower'}
 pairs.append({'review_unit_id':rid,'page':page,'source_group':rep.source_group,'A':A,'B':B,'A_sha256':sha(A),'B_sha256':sha(B),'side_lock':base64.urlsafe_b64encode(json.dumps(side).encode()).decode(),'overlay':rep.overlay_path,'difference':rep.difference_path,'v78':rep.v78_path,'observed':rep.observed_path,'represented_transitions':len(g)})
 mapping.extend({'review_unit_id':rid,**x} for x in g.to_dict('records'))
blind=pd.DataFrame(pairs);transition_map=pd.DataFrame(mapping)
if blind.review_unit_id.duplicated().any():stop('deduplication','Duplicate unique pair IDs')
csv(blind,TABLES/'blinded_review_manifest.csv');csv(transition_map,TABLES/'review_transition_map.csv')
print('Page transitions:',len(resolved));print('Unique whole-page pairs:',len(blind));print('Additional identical pairs removed:',len(resolved)-len(blind))


## 6. Guided reviewer

First compare A and B as corrected whole pages. The blinded first pass is saved before diagnostics appear.

**Harm:** P0 none; P1 barely visible/cosmetic; P2 clear local defect; P3 serious/repeated interference; P4 catastrophic/page-level; U cannot judge.

**Benefit:** B0 none; B1 barely visible/local; B2 clear local improvement; B3 substantial/repeated improvement; B4 major page-level improvement; U cannot judge.


In [ ]:
import gradio as gr
R1=REVIEW/'perceptual_review_round1.csv';R2=REVIEW/'diagnostic_review_round2.csv'
C1=['review_unit_id','visibility','preference','harm','benefit','scope','musical_risk','prominence','action','confidence','note','utc','locked','record_sha256'];C2=['review_unit_id','mechanism','resolved_harm','resolved_benefit','note','utc','record_sha256']
def readr(p,cols):
 if not Path(p).exists():return pd.DataFrame(columns=cols)
 d=pd.read_csv(p,keep_default_na=False)
 for x in cols:
  if x not in d:d[x]=''
 return d[cols]
def save(p,cols,d):
 d['record_sha256']=hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest();z=readr(p,cols);z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])];csv(pd.concat([z,pd.DataFrame([d])],ignore_index=True)[cols],p);return d['record_sha256']
def pending():
 done=set(readr(R2,C2).review_unit_id.astype(str));return [i for i,x in enumerate(blind.review_unit_id.astype(str)) if x not in done]
def im(x):return str(x) if str(x) and Path(str(x)).is_file() else None
def view(i):
 i=int(max(0,min(int(i),len(blind)-1)));r=blind.iloc[i];status=f'### Review {i+1} of {len(blind)}  \nCompleted: {len(readr(R2,C2))}  \nRepresents {int(r.represented_transitions)} transition(s) with this exact pair.';return i,im(r.A),im(r.B),status,r.review_unit_id
def first(i,visibility,preference,harm,benefit,scope,risk,prominence,action,confidence,note):
 vals=[visibility,preference,harm,benefit,scope,risk,prominence,action,confidence]
 if any(x in ['',None] for x in vals):raise gr.Error('Complete all numbered first-pass questions.')
 r=blind.iloc[int(i)];d={'review_unit_id':r.review_unit_id,'visibility':visibility,'preference':preference,'harm':harm,'benefit':benefit,'scope':scope,'musical_risk':risk,'prominence':prominence,'action':action,'confidence':confidence,'note':str(note),'utc':now(),'locked':True};h=save(R1,C1,d);return f'First pass locked ({h[:12]}). Use diagnostics to explain it.',gr.update(visible=False),gr.update(visible=True),im(r.overlay),im(r.difference),im(r.v78),im(r.observed)
def diagnostic(i,mechanism,harm,benefit,note):
 if any(x in ['',None] for x in [mechanism,harm,benefit]):raise gr.Error('Complete all diagnostic questions.')
 r=blind.iloc[int(i)];h=save(R2,C2,{'review_unit_id':r.review_unit_id,'mechanism':mechanism,'resolved_harm':harm,'resolved_benefit':benefit,'note':str(note),'utc':now()});p=pending();nxt=next((x for x in p if x>int(i)),p[0] if p else int(i));return view(nxt)+(f'Saved ({h[:12]}). Loading {nxt+1}.',gr.update(visible=True),gr.update(visible=False))
GUIDE='\n'.join(['## What to do','1. Compare Page A and Page B as whole corrected pages.','2. Record visibility and overall preference.','3. Rate harm and benefit using the displayed rubrics.','4. Save the blind first pass.','5. Use diagnostics to explain it, save, and continue.','','If the pages look identical choose: No visible difference; Neither; P0; B0; No visible defect; None; None; Accept either.'])
PG='P0 none | P1 barely visible | P2 clear local | P3 serious/repeated | P4 catastrophic/page-level | U cannot judge';BG='B0 none | B1 barely visible | B2 clear local | B3 substantial/repeated | B4 major page-level | U cannot judge'
with gr.Blocks(title='Notebook 38.5 guided reviewer') as app:
 gr.Markdown('# Notebook 38.5 — guided blinded whole-page comparison');gr.Markdown(GUIDE);state=gr.State(pending()[0] if pending() else 0);unit=gr.Textbox(visible=False);status=gr.Markdown()
 with gr.Row():A=gr.Image(type='filepath',label='Page A',height=720);B=gr.Image(type='filepath',label='Page B',height=720)
 with gr.Column(visible=True) as p1:
  gr.Markdown('## First pass — no diagnostics yet');visibility=gr.Radio(['No visible difference','Visible difference','Cannot tell'],label='1. Can you see any difference?');preference=gr.Radio(['A','B','Neither — equivalent','Mixed','Cannot tell'],label='2. Which page is preferable overall?');gr.Markdown('**Harm rubric:** '+PG);harm=gr.Radio(['P0','P1','P2','P3','P4','U'],label='3. Harm: retained annotation or worse musical result');gr.Markdown('**Benefit rubric:** '+BG);benefit=gr.Radio(['B0','B1','B2','B3','B4','U'],label='4. Benefit: genuine notation preserved better');scope=gr.Dropdown(['No visible defect','Single pixel/fragment','One local component','One staff/system','Repeated across page','Page-wide','Cannot tell'],label='5. Largest visible scope');risk=gr.Dropdown(['None','Staff line','Notehead','Stem or beam','Accidental','Rest','Articulation','Dynamic','Text or lyric','Barline or repeat','Other','Cannot tell'],label='6. Musical element affected');prominence=gr.Radio(['None','Faint','Moderate','Prominent'],label='7. Visual prominence');action=gr.Radio(['Accept either','Prefer A','Prefer B','Needs manual correction','Reject this threshold transition','Cannot tell'],label='8. Recommended action');confidence=gr.Radio(['Low','Medium','High'],label='9. Confidence');note=gr.Textbox(label='10. Optional note',lines=3);b1=gr.Button('Save first pass and reveal diagnostics',variant='primary')
 with gr.Column(visible=False) as p2:
  gr.Markdown('## Diagnostic pass — first pass is locked')
  with gr.Row():overlay=gr.Image(label='Newly preserved overlay');difference=gr.Image(label='Difference from registered V8.2');v78=gr.Image(label='Frozen V7.8');observed=gr.Image(label='Observed page')
  mechanism=gr.Dropdown(['Genuine print recovery','Retained annotation','Mixed print and annotation','Registration ambiguity','Truth-boundary sensitivity','Display artefact','No material difference','Other'],label='11. Best explanation');gr.Markdown('**Resolved harm:** '+PG);rh=gr.Radio(['P0','P1','P2','P3','P4','U'],label='12. Final harm');gr.Markdown('**Resolved benefit:** '+BG);rb=gr.Radio(['B0','B1','B2','B3','B4','U'],label='13. Final benefit');dn=gr.Textbox(label='14. Diagnostic note',lines=3);b2=gr.Button('Save diagnostic and load next review',variant='primary')
 message=gr.Markdown();app.load(view,[state],[state,A,B,status,unit]);b1.click(first,[state,visibility,preference,harm,benefit,scope,risk,prominence,action,confidence,note],[message,p1,p2,overlay,difference,v78,observed]);b2.click(diagnostic,[state,mechanism,rh,rb,dn],[state,A,B,status,unit,message,p1,p2])
app.launch(share=True,debug=False)


## 7. Completion, summary and final package


In [ ]:
r1=readr(R1,C1);r2=readr(R2,C2);expected=set(blind.review_unit_id.astype(str));done1=set(r1.review_unit_id.astype(str));done2=set(r2.review_unit_id.astype(str));coverage=pd.DataFrame({'review_unit_id':sorted(expected),'round1_complete':[x in done1 for x in sorted(expected)],'round2_complete':[x in done2 for x in sorted(expected)]});csv(coverage,AUDIT/'review_coverage.csv')
if not coverage[['round1_complete','round2_complete']].all().all():stop('completion','Review incomplete',{'expected':len(expected),'round1':len(done1&expected),'round2':len(done2&expected)})
unique_reviews=blind.merge(r1,on='review_unit_id',validate='one_to_one').merge(r2,on='review_unit_id',validate='one_to_one');transition_reviews=transition_map.merge(r1,on='review_unit_id',validate='many_to_one').merge(r2,on='review_unit_id',validate='many_to_one');csv(unique_reviews,TABLES/'completed_unique_pair_reviews.csv');csv(transition_reviews,TABLES/'completed_transition_reviews.csv');csv(pd.DataFrame([{'reviewer_count':1,'agreement_status':'not_estimable_single_reviewer'}]),TABLES/'reviewer_agreement.csv')
hn={'P0':0,'P1':1,'P2':2,'P3':3,'P4':4};bn={'B0':0,'B1':1,'B2':2,'B3':3,'B4':4};transition_reviews['harm_n']=transition_reviews.resolved_harm.map(hn);transition_reviews['benefit_n']=transition_reviews.resolved_benefit.map(bn)
if transition_reviews[['harm_n','benefit_n']].isna().any().any():stop('analysis','Final U ratings remain unresolved')
summary=pd.DataFrame([{'threshold':t,'transitions':len(z),'unique_pairs':z.review_unit_id.nunique(),'P2_plus':int(z.harm_n.ge(2).sum()),'P3_plus':int(z.harm_n.ge(3).sum()),'P4':int(z.harm_n.eq(4).sum()),'B2_plus':int(z.benefit_n.ge(2).sum())} for t in THRESHOLDS for z in [transition_reviews[transition_reviews.lo.astype(float).ge(t)]]]);csv(summary,TABLES/'perceptual_D3_D8_results.csv');csv(summary,TABLES/'envelope_relaxation_frontier.csv')
plt.figure(figsize=(9,5));plt.plot(summary.threshold,summary.P2_plus,marker='o',label='P2+');plt.plot(summary.threshold,summary.B2_plus,marker='o',label='B2+');plt.xlabel('Threshold');plt.ylabel('Transition records');plt.legend();plt.tight_layout();plt.savefig(VIS/'perceptual_summary.png',dpi=160);plt.close()
status='single-reviewer-exploratory-with-provenance-warning' if manifest_summary['warnings'] else 'single-reviewer-exploratory';verdict={'version':VERSION,'status':status,'component_rows':len(raw),'page_transitions':len(transitions),'unique_pair_reviews':len(blind),'production_authority':False,'threshold_adoption_authority':False,'source_lock_sha256':source_lock_sha256};js(OUT/'notebook38_verdict.json',verdict);text(OUT/'executive_summary.txt',json.dumps(verdict,indent=2)+'\n')
items=[OUT/'protocol_lock.json',OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',TABLES/'registered_D3_D8_results.csv',TABLES/'component_queue_source_rows.csv',TABLES/'page_transition_manifest.csv',TABLES/'blinded_review_manifest.csv',TABLES/'review_transition_map.csv',TABLES/'completed_unique_pair_reviews.csv',TABLES/'completed_transition_reviews.csv',TABLES/'reviewer_agreement.csv',TABLES/'perceptual_D3_D8_results.csv',R1,R2,VIS/'perceptual_summary.png',AUDIT/'notebook38_source_input_lock.csv',AUDIT/'nb37_manifest_verification.csv',AUDIT/'nb37_manifest_verification_summary.json',AUDIT/'whole_page_asset_resolution.csv',AUDIT/'missing_review_assets.csv',AUDIT/'review_coverage.csv',REF/'notebook37_experiment_identity.json',REF/'source_v82_record.json']
missing=[str(p) for p in items if not p.exists()]
if missing:stop('package','Missing outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
 rel=src.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'bytes':dst.stat().st_size,'sha256':sha(dst)})
text(UPLOAD/'README.txt','Notebook 38.5 deduplicated guided whole-page review; single-reviewer exploratory only.\n');rows.append({'relative_path':'README.txt','bytes':(UPLOAD/'README.txt').stat().st_size,'sha256':sha(UPLOAD/'README.txt')});csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv');bundle=OUT/'notebook38_5_upload_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
js(OUT/'final_package_record.json',{'bundle':str(bundle),'sha256':sha(bundle),'utc':now()});print(json.dumps(verdict,indent=2));print(bundle)


## Interpretation boundary
Notebook 37's persisted corrected pages are the review evidence. One review covers each unique exact page pair and is mapped back to every represented adjacent-threshold transition. Diagnostics appear only after the blind first pass is locked. The result cannot adopt or promote a threshold.
